# Pesquisa adaptativa de fibras — graus ímpares HRSBF

Alvos iniciais: quínticas e sépticas homogêneas RS em 16 variáveis. Cada rodada avalia todas as fibras, a derivada na direção de todos os uns, funções esparsas e mutações dos candidatos mais próximos das condições necessárias. Uma cobertura aprendida é testada em candidatos novos antes de sugerir novas fibras.

**Resultados são amostras, não uma prova universal.** O caminho CUDA ainda precisa passar pelos controles dentro da sua sessão. Selecione uma GPU no menu **Ambiente de execução → Alterar tipo de ambiente de execução**, conecte e execute as células. A A100 é opcional; outra GPU CUDA também serve. A primeira execução tem limite de 30 minutos por alvo. O notebook não instala pacotes nem depende de downloads do repositório.

Os checkpoints ficam no armazenamento temporário do Colab. Baixe o ZIP ao final; uma reinicialização pode apagá-los.


In [ ]:
import sys, platform, numpy as np
try:
    import torch
except ImportError:
    raise RuntimeError("PyTorch não está disponível neste ambiente.")
print("Python", sys.version, "NumPy", np.__version__, "PyTorch", torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("GPU não conectada: selecione uma GPU no Colab e conecte o ambiente.")
print("GPU OBSERVADA:", torch.cuda.get_device_name(0))


In [ ]:
from pathlib import Path
Path('pesquisa_fibras_loop.py').write_text('#!/usr/bin/env python3\n"""Bounded adaptive search: necessary bent conditions for odd-degree HRSBF.\n\nNumPy CPU and optional PyTorch CUDA backends. No claim of exhaustive proof.\nAll CUDA GEMMs use float32, TF32 disabled, with exact integer dot products\nbounded by the number of orbit coefficients. CPU audits parity independently.\n"""\nimport argparse\nimport hashlib\nimport itertools\nimport json\nimport time\nfrom pathlib import Path\n\nimport numpy as np\n\n\ndef orbit_basis(n, degree):\n    unseen = set(itertools.combinations(range(n), degree))\n    orbits = []\n    while unseen:\n        representative = min(unseen)\n        orbit = sorted({tuple(sorted((i + shift) % n for i in representative))\n                        for shift in range(n)})\n        unseen.difference_update(orbit)\n        orbits.append((representative, orbit))\n    return sorted(orbits)\n\n\ndef make_tables(n, degree):\n    orbits = orbit_basis(n, degree)\n    inputs = np.arange(1 << n, dtype=np.uint32)\n    basis = np.zeros((len(orbits), len(inputs)), dtype=np.uint8)\n    for row, (_, orbit) in enumerate(orbits):\n        for support in orbit:\n            mask = sum(1 << i for i in support)\n            basis[row] ^= ((inputs & mask) == mask).astype(np.uint8)\n    h = n // 2\n    u = np.arange(1 << h, dtype=np.uint32)\n    z = u.copy()\n    indices = (u[None, :] | ((u[None, :] ^ z[:, None]) << h)).astype(np.int64)\n    if np.any(basis[:, indices[0]]):\n        raise RuntimeError(\'Odd-degree diagonal cancellation failed\')\n    return orbits, basis, indices\n\n\nclass Evaluator:\n    def __init__(self, basis, indices, mode):\n        if len(basis) >= 2 ** 24:\n            raise ValueError(\'Dot products exceed the exact float32 integer range\')\n        self.basis = basis\n        self.indices = indices\n        self.backend = \'numpy-cpu\'\n        self.gpu = None\n        self.torch = None\n        if mode != \'cpu\':\n            try:\n                import torch\n                if torch.cuda.is_available():\n                    self.torch = torch\n                    torch.backends.cuda.matmul.allow_tf32 = False\n                    torch.backends.cudnn.allow_tf32 = False\n                    torch.set_float32_matmul_precision(\'highest\')\n                    self.basis_gpu = torch.tensor(basis, dtype=torch.float32, device=\'cuda\')\n                    self.indices_gpu = torch.tensor(indices.reshape(-1), device=\'cuda\')\n                    self.backend = \'torch-cuda-float32-TF32-disabled\'\n                    self.gpu = torch.cuda.get_device_name(0)\n                elif mode == \'cuda\':\n                    raise RuntimeError(\'CUDA requested but no GPU is connected\')\n            except ImportError:\n                if mode == \'cuda\':\n                    raise RuntimeError(\'CUDA backend needs PyTorch\')\n        self.basis_float = basis.astype(np.float32) if self.torch is None else None\n\n    def evaluate(self, coefficients):\n        count = len(coefficients)\n        hsize = self.indices.shape[0]\n        if self.torch is None:\n            sums = coefficients.astype(np.float32) @ self.basis_float\n            tables = sums.astype(np.int32) & 1\n            weights = tables[:, self.indices.reshape(-1)].reshape(count, hsize, hsize).sum(axis=2)\n            derivative_weights = np.bitwise_xor(tables, tables[:, ::-1]).sum(axis=1)\n        else:\n            t = self.torch\n            with t.no_grad():\n                c = t.tensor(coefficients, dtype=t.float32, device=\'cuda\')\n                sums = c @ self.basis_gpu\n                tables_gpu = sums.to(t.int32).bitwise_and(1)\n                weights = tables_gpu[:, self.indices_gpu].reshape(count, hsize, hsize).sum(dim=2).cpu().numpy()\n                derivative_weights = tables_gpu.bitwise_xor(tables_gpu.flip((1,))).sum(dim=1).cpu().numpy()\n                # Only rows needed for CPU controls are copied below.\n                tables = tables_gpu[:min(4, count)].cpu().numpy()\n        # Exact integer CPU XOR validates each round\'s first rows independently\n        # of matrix multiplication, including before and after CUDA transfer.\n        for j in range(min(4, count)):\n            active = self.basis[coefficients[j].astype(bool)]\n            direct = np.bitwise_xor.reduce(active, axis=0) if len(active) else np.zeros(self.basis.shape[1], dtype=np.uint8)\n            if not np.array_equal(tables[j], direct):\n                raise RuntimeError(\'Backend disagrees with independent XOR evaluator\')\n            direct_weights = direct[self.indices].sum(axis=1)\n            if not np.array_equal(weights[j], direct_weights):\n                raise RuntimeError(\'Fiber weights disagree with CPU reconstruction\')\n            if int(derivative_weights[j]) != int(np.bitwise_xor(direct, direct[::-1]).sum()):\n                raise RuntimeError(\'All-ones derivative disagrees with CPU reconstruction\')\n        return weights, derivative_weights\n\n\ndef greedy_cover(failures):\n    remaining = np.ones(len(failures), dtype=bool)\n    selected = []\n    while remaining.any():\n        counts = failures[remaining].sum(axis=0)\n        best = int(counts.argmax())\n        if counts[best] == 0:\n            break\n        selected.append({\'z\': best + 1, \'new_exclusions\': int(counts[best])})\n        remaining &= ~failures[:, best]\n    return selected, int(remaining.sum())\n\n\ndef draw_batch(rng, width, batch, elites, round_id):\n    coefficients = rng.integers(0, 2, size=(batch, width), dtype=np.uint8)\n    kinds = np.zeros(batch, dtype=np.uint8)  # fresh dense / sparse / mutated\n    sparse_start = batch // 2\n    mutation_start = 3 * batch // 4\n    coefficients[sparse_start:mutation_start] = 0\n    kinds[sparse_start:mutation_start] = 1\n    for row in range(sparse_start, mutation_start):\n        support = rng.choice(width, size=int(rng.integers(1, min(width, 12) + 1)), replace=False)\n        coefficients[row, support] = 1\n    if len(elites):\n        kinds[mutation_start:] = 2\n        for row in range(mutation_start, batch):\n            coefficients[row] = elites[int(rng.integers(len(elites)))].copy()\n            flips = rng.choice(width, size=int(rng.integers(1, min(width, 8) + 1)), replace=False)\n            coefficients[row, flips] ^= 1\n    empty = np.flatnonzero(coefficients.sum(axis=1) == 0)\n    coefficients[empty, 0] = 1\n    return coefficients, kinds\n\n\ndef candidate_record(coefficients, weights, derivative_weight, representatives):\n    bad = np.flatnonzero(weights[1:] != len(weights) // 2)\n    mask = sum(int(bit) << i for i, bit in enumerate(coefficients))\n    witness = None if not len(bad) else {\'z\': int(bad[0] + 1), \'weight\': int(weights[bad[0] + 1])}\n    return {\'coefficient_mask\': hex(mask),\n            \'active_orbit_representatives\': [list(representatives[i]) for i in np.flatnonzero(coefficients)],\n            \'allones_derivative_weight\': int(derivative_weight),\n            \'unbalanced_nonzero_fibers\': int(len(bad)),\n            \'exclusion_witness\': witness,\n            \'status\': \'EXCLUDED_BY_FIBER\' if witness else \'NECESSARY_CONDITIONS_ONLY\'}\n\n\ndef validate_sample_cover(folder, draws=8192, seed=90420261006, backend=\'cpu\'):\n    """New dense holdout; test learned cover and inspect its survivors."""\n    folder = Path(folder)\n    summary = json.loads((folder / \'summary.json\').read_text())\n    n, degree = summary[\'config\'][\'n\'], summary[\'config\'][\'degree\']\n    orbits, basis, indices = make_tables(n, degree)\n    cover = [record[\'z\'] for record in summary[\'global_sample_cover\']]\n    if not cover:\n        return {\'status\': \'NO_LEARNED_COVER\'}\n    rng = np.random.default_rng(seed)\n    coefficients = rng.integers(0, 2, (draws, len(basis)), dtype=np.uint8)\n    # This short cover evaluates only the selected fibers; all products are\n    # exact integer values <= number of generators in float32.\n    selected_basis = basis[:, indices[cover].reshape(-1)].astype(np.float32)\n    weights = ((coefficients.astype(np.float32) @ selected_basis).astype(np.int32) & 1)\n    weights = weights.reshape(draws, len(cover), 1 << (n//2)).sum(axis=2)\n    survivors = np.flatnonzero(np.all(weights == (1 << (n//2-1)), axis=1))\n    records = []\n    additional_cover = []\n    if len(survivors):\n        engine = Evaluator(basis, indices, backend)\n        all_weights, derivative = engine.evaluate(coefficients[survivors])\n        failure = all_weights[:, 1:] != (1 << (n//2-1))\n        additional_cover, still_uncovered = greedy_cover(failure)\n        records = [candidate_record(coefficients[survivors[j]], all_weights[j], derivative[j],\n                                    [r for r, _ in orbits]) for j in range(min(4, len(survivors)))]\n    else:\n        still_uncovered = 0\n    result = {\'fresh_seed\': seed, \'fresh_dense_draws\': draws, \'learned_fibers\': cover,\n              \'cover_exclusions\': draws - len(survivors), \'cover_survivors\': len(survivors),\n              \'additional_fibers_covering_recorded_survivors\': additional_cover,\n              \'still_uncovered_by_all_fibers\': still_uncovered,\n              \'survivor_examples\': records,\n              \'conclusion\': \'A learned sample cover must be retested on independent draws; it is not a universal proof.\'}\n    (folder / \'holdout.json\').write_text(json.dumps(result, indent=2) + \'\\n\')\n    return result\n\n\ndef run(n=16, degree=5, rounds=4, batch=256, backend=\'auto\', seed=20261006,\n        output=\'resultados_fibras_loop\', max_seconds=1800):\n    if n not in (8, 16) or degree % 2 != 1 or degree < 3 or degree > n:\n        raise ValueError(\'Supported targets: n=8 or 16 and homogeneous odd degree >=3\')\n    if rounds < 1 or batch < 8 or max_seconds < 1:\n        raise ValueError(\'Positive bounded rounds, time and batch>=8 required\')\n    folder = Path(output)\n    folder.mkdir(parents=True, exist_ok=True)\n    config = dict(n=n, degree=degree, rounds=rounds, batch=batch, backend=backend,\n                  seed=seed, max_seconds=max_seconds)\n    started = time.monotonic()\n    orbits, basis, indices = make_tables(n, degree)\n    representatives = [rep for rep, _ in orbits]\n    engine = Evaluator(basis, indices, backend)\n    rng = np.random.default_rng(seed)\n    elites = np.empty((0, len(basis)), dtype=np.uint8)\n    previous_cover = []\n    history = []\n    failures_history = []\n    potential_candidates = []\n    best_records = []\n    print(json.dumps({\'backend\': engine.backend, \'gpu\': engine.gpu, \'basis_orbits\': len(basis), \'config\': config}), flush=True)\n    for round_id in range(rounds):\n        if round_id > 0 and time.monotonic() - started > max_seconds:\n            break\n        coefficients, kinds = draw_batch(rng, len(basis), batch, elites, round_id)\n        weights, derivative = engine.evaluate(coefficients)\n        expected_fiber_weight = (1 << (n // 2)) // 2\n        failures = weights[:, 1:] != expected_fiber_weight\n        derivative_balanced = derivative == (1 << n) // 2\n        complementary_balanced = ~failures[:, -1]\n        both = derivative_balanced & complementary_balanced\n        all_balanced = ~failures.any(axis=1)\n        # Evaluate a cover learned in the PREVIOUS round on a fresh dense cohort.\n        holdout = kinds == 0\n        if previous_cover:\n            cols = [record[\'z\'] - 1 for record in previous_cover]\n            validation_exclusions = int(failures[holdout][:, cols].any(axis=1).sum())\n        else:\n            validation_exclusions = None\n        cover, uncovered = greedy_cover(failures)\n        order = np.argsort(failures.sum(axis=1), kind=\'stable\')[:min(32, batch)]\n        elites = coefficients[order].copy()\n        for row in np.flatnonzero(all_balanced & derivative_balanced):\n            potential_candidates.append(candidate_record(coefficients[row], weights[row], derivative[row], representatives))\n        best_records = [candidate_record(coefficients[row], weights[row], derivative[row], representatives) for row in order[:4]]\n        record = {\'round\': round_id + 1, \'tested\': batch,\n                  \'cohorts\': {name: int((kinds == index).sum()) for index, name in enumerate((\'fresh_dense\', \'fresh_sparse\', \'elite_mutations\'))},\n                  \'derivative_balanced\': int(derivative_balanced.sum()),\n                  \'complementary_balanced\': int(complementary_balanced.sum()),\n                  \'both_balanced\': int(both.sum()),\n                  \'both_excluded_by_other_fibers\': int((both & failures.any(axis=1)).sum()),\n                  \'all_nonzero_fibers_balanced\': int(all_balanced.sum()),\n                  \'all_fibers_and_derivative_balanced\': int((all_balanced & derivative_balanced).sum()),\n                  \'greedy_sample_cover\': cover, \'uncovered_by_all_fibers\': uncovered,\n                  \'previous_cover_fresh_dense_exclusions\': validation_exclusions,\n                  \'fresh_dense_validation_size\': int(holdout.sum()),\n                  \'closest_candidate_unbalanced_fiber_count\': int(failures.sum(axis=1).min())}\n        history.append(record)\n        failures_history.append(failures)\n        previous_cover = cover\n        # Per-round data persists before the next round; Colab can export it.\n        np.savez_compressed(folder / f\'round_{round_id+1:04d}.npz\',\n                            coefficients=coefficients, kinds=kinds, weights=weights,\n                            derivative_weights=derivative)\n        (folder / \'checkpoint.json\').write_text(json.dumps({\'config\': config,\n            \'completed_rounds\': round_id + 1, \'rng_state\': rng.bit_generator.state,\n            \'latest_cover\': cover, \'warning\': \'This is sampled evidence, not a universal certificate.\'}, indent=2) + \'\\n\')\n        print(json.dumps(record), flush=True)\n    failure_matrix = np.vstack(failures_history)\n    global_cover, global_uncovered = greedy_cover(failure_matrix)\n    summary = {\'status\': \'COMPLETED\', \'config\': config, \'actual_backend\': engine.backend,\n               \'gpu\': engine.gpu, \'basis_orbits\': len(basis),\n               \'basis_sha256\': hashlib.sha256(basis.tobytes()).hexdigest(),\n               \'representatives\': [list(rep) for rep in representatives],\n               \'tested_total\': sum(row[\'tested\'] for row in history),\n               \'rounds_completed\': len(history), \'rounds\': history,\n               \'global_sample_cover\': global_cover,\n               \'uncovered_by_all_fibers\': global_uncovered,\n               \'potential_candidates\': potential_candidates,\n               \'latest_closest_candidates\': best_records,\n               \'elapsed_seconds\': round(time.monotonic() - started, 3),\n               \'limitations\': [\'Adaptive samples can repeat candidates; tested_total counts draws.\',\n                              \'A cover excludes only the recorded samples, not the full coefficient space.\',\n                              \'No result is a proof of quintic nonexistence or proof-assistant certification.\',\n                              \'GPU execution is reported only when the actual backend is CUDA.\']}\n    (folder / \'summary.json\').write_text(json.dumps(summary, indent=2) + \'\\n\')\n    (folder / \'next_hypotheses.json\').write_text(json.dumps({\n        \'classification\': \'research proposals, not theorems\',\n        \'proposals\': [\n            {\'name\': \'small_fiber_cover\', \'fibers\': [r[\'z\'] for r in global_cover],\n             \'next_test\': \'Evaluate on a new independent seed and all sparse supports up to a chosen bound.\'},\n            {\'name\': \'conditional_derivative_fiber_obstruction\',\n             \'next_test\': \'Search specifically among candidates satisfying all-ones derivative and complementary balance.\'},\n            {\'name\': \'universal_certificate\',\n             \'next_test\': \'Derive a faithful exact constraint model and independently verify exclusions; sample cover alone is insufficient.\'}]}, indent=2) + \'\\n\')\n    return summary\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--n\', type=int, default=16)\n    parser.add_argument(\'--degree\', type=int, default=5)\n    parser.add_argument(\'--rounds\', type=int, default=4)\n    parser.add_argument(\'--batch\', type=int, default=256)\n    parser.add_argument(\'--backend\', choices=(\'auto\', \'cpu\', \'cuda\'), default=\'auto\')\n    parser.add_argument(\'--seed\', type=int, default=20261006)\n    parser.add_argument(\'--output\', default=\'resultados_fibras_loop\')\n    parser.add_argument(\'--max-seconds\', type=int, default=1800)\n    args = parser.parse_args()\n    result = run(**vars(args))\n    print(json.dumps({k: result[k] for k in (\'actual_backend\', \'tested_total\', \'rounds_completed\', \'uncovered_by_all_fibers\', \'elapsed_seconds\')}, indent=2))\n', encoding="utf-8")
Path('verify_fibras_loop.py').write_text('#!/usr/bin/env python3\n"""Independent direct-ANF checks and exhaustive small-dimensional controls."""\nimport argparse\nimport itertools\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nfrom pesquisa_fibras_loop import Evaluator, make_tables\n\n\ndef fwht(table):\n    spectrum = 1 - 2 * np.asarray(table, dtype=np.int32)\n    step = 1\n    while step < len(spectrum):\n        for start in range(0, len(spectrum), 2 * step):\n            left = spectrum[start:start+step].copy()\n            right = spectrum[start+step:start+2*step].copy()\n            spectrum[start:start+step] = left + right\n            spectrum[start+step:start+2*step] = left - right\n        step *= 2\n    return spectrum\n\n\ndef directly_evaluate(coefficients, orbits, x):\n    value = 0\n    for coefficient, (_, orbit) in zip(coefficients, orbits):\n        if coefficient:\n            for support in orbit:\n                value ^= int(all(x >> i & 1 for i in support))\n    return value\n\n\ndef small_controls(backend):\n    records = []\n    for degree in (3, 5):\n        n = 8\n        orbits, basis, indices = make_tables(n, degree)\n        coefficients = np.asarray([[c >> i & 1 for i in range(len(orbits))]\n                                   for c in range(1, 1 << len(orbits))], dtype=np.uint8)\n        engine = Evaluator(basis, indices, backend)\n        weights, derivatives = engine.evaluate(coefficients)\n        bent_count = semibent_count = 0\n        for row, coeff in enumerate(coefficients):\n            table = np.asarray([directly_evaluate(coeff, orbits, x) for x in range(256)], dtype=np.uint8)\n            if not np.array_equal(weights[row], table[indices].sum(axis=1)):\n                raise RuntimeError(\'Exhaustive direct-ANF fiber comparison failed\')\n            if derivatives[row] != np.bitwise_xor(table, table[::-1]).sum():\n                raise RuntimeError(\'Exhaustive direct-ANF derivative comparison failed\')\n            spectrum = fwht(table)\n            bent_count += bool(np.all(np.abs(spectrum) == 16))\n            semibent_count += set(map(abs, spectrum.tolist())) == {0, 32}\n        if bent_count or semibent_count:\n            raise RuntimeError(\'Small control counts disagreed with recorded results\')\n        records.append({\'n\': n, \'degree\': degree, \'nonzero_functions\': len(coefficients),\n                        \'bent\': bent_count, \'semibent_0_32\': semibent_count,\n                        \'actual_backend\': engine.backend, \'status\': \'PASS\'})\n    # Positive bent control: q(x)=sum x_i*x_{i+4}. This tests the Walsh routine.\n    positive = [sum(((x >> i) & 1) * ((x >> (i+4)) & 1) for i in range(4)) % 2\n                for x in range(256)]\n    if not np.all(np.abs(fwht(positive)) == 16):\n        raise RuntimeError(\'Positive bent control failed\')\n    return {\'status\': \'PASS\', \'controls\': records, \'quadratic_bent_positive_control\': \'PASS\'}\n\n\ndef verify_round(folder, round_number):\n    folder = Path(folder)\n    summary = json.loads((folder / \'summary.json\').read_text())\n    config = summary[\'config\']\n    n, degree = config[\'n\'], config[\'degree\']\n    orbits, basis, indices = make_tables(n, degree)\n    data = np.load(folder / f\'round_{round_number:04d}.npz\')\n    coefficients, weights = data[\'coefficients\'], data[\'weights\']\n    failures = weights[:, 1:] != (1 << (n//2-1))\n    closest = int(np.argmin(failures.sum(axis=1)))\n    bad = np.flatnonzero(failures[closest])\n    if not len(bad):\n        return {\'status\': \'NEEDS_FULL_SPECTRAL_CHECK\', \'row\': closest}\n    z = int(bad[0] + 1)\n    # Independent monomial evaluator reconstructs the entire witness fiber.\n    weight = sum(directly_evaluate(coefficients[closest], orbits, int(x)) for x in indices[z])\n    if weight != int(weights[closest, z]) or weight == (1 << (n//2-1)):\n        raise RuntimeError(\'Direct ANF witness verification failed\')\n    return {\'status\': \'PASS\', \'row\': closest, \'z\': z, \'direct_anf_weight\': weight,\n            \'balanced_weight_required\': 1 << (n//2-1)}\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--backend\', choices=(\'cpu\', \'cuda\', \'auto\'), default=\'cpu\')\n    parser.add_argument(\'--folder\')\n    parser.add_argument(\'--round\', type=int, default=1)\n    parser.add_argument(\'--output\', default=\'verificacao_loop.json\')\n    args = parser.parse_args()\n    result = small_controls(args.backend)\n    if args.folder:\n        result[\'round_witness\'] = verify_round(args.folder, args.round)\n    Path(args.output).write_text(json.dumps(result, indent=2) + \'\\n\')\n    print(json.dumps(result, indent=2))\n', encoding="utf-8")
print("Código autocontido gravado; não foram baixadas dependências.")


## Validação obrigatória antes da campanha

A enumeração completa das 127 funções não nulas em n=8, grau 3 e grau 5, é conferida por avaliação direta da ANF. Um controle bent quadrático verifica a transformação de Walsh. Divergências encerram a execução.


In [ ]:
import json
from verify_fibras_loop import small_controls, verify_round
from pesquisa_fibras_loop import run, validate_sample_cover
cpu_control = small_controls("cpu")
gpu_control = small_controls("cuda")
assert [(c["n"], c["degree"], c["bent"], c["semibent_0_32"]) for c in cpu_control["controls"]] == [(c["n"], c["degree"], c["bent"], c["semibent_0_32"]) for c in gpu_control["controls"]]
Path("validacao_cpu_gpu.json").write_text(json.dumps({"cpu":cpu_control,"gpu":gpu_control},indent=2))
print(json.dumps(gpu_control,indent=2))


## Campanha em GPU

Cada alvo tem 100 rodadas de 512 candidatos e limite de 1.800 segundos. Isso conta sorteios; pode haver repetições. Para ampliar, mude RODADAS e LIMITE_SEGUNDOS conscientemente. O teste retém as melhores tentativas e usa suas mutações, mas mantém coortes novas para evitar validar a hipótese apenas nos dados usados para formulá-la.


In [ ]:
import time
RODADAS = 100
LOTE = 512
LIMITE_SEGUNDOS = 1800
ALVOS = [(16, 5, 20261006), (16, 7, 20261007)]
carimbo = time.strftime("%Y%m%d_%H%M%S")
resultados = []
for n, grau, semente in ALVOS:
    pasta = f"campanha_{carimbo}_n{n}_d{grau}"
    resultado = run(n=n, degree=grau, rounds=RODADAS, batch=LOTE, backend="cuda", seed=semente, output=pasta, max_seconds=LIMITE_SEGUNDOS)
    testemunha = verify_round(pasta, resultado["rounds_completed"])
    validacao = validate_sample_cover(pasta, draws=8192, seed=semente+987654321, backend="cuda")
    Path(pasta, "testemunha_anf.json").write_text(json.dumps(testemunha,indent=2))
    resultados.append({"pasta":pasta,"resultado":resultado,"validacao":validacao})
    print("ALVO FINALIZADO:", n, grau, "GPU:", resultado["gpu"], "sorteios:",resultado["tested_total"], "sobreviventes da cobertura no holdout:",validacao.get("cover_survivors"))


## Próximas hipóteses e exportação

Uma cobertura que falhou no holdout deve ser ampliada com novas fibras. As propostas geradas são perguntas de pesquisa, não teoremas. Para um candidato que satisfaça todas as condições, ainda falta o espectro completo de Walsh. Guarde os NPZ: eles permitem reconstruir os coeficientes e cada peso calculado.


In [ ]:
import zipfile
# Esta célula também exporta checkpoints se a campanha foi interrompida.
carimbo_exportacao = globals().get("carimbo", "parcial")
pastas = list(Path(".").glob(f"campanha_{carimbo_exportacao}_n*_d*"))
for pasta in pastas:
    propostas = pasta / "next_hypotheses.json"
    if propostas.exists(): print(pasta, propostas.read_text())
zip_path = Path(f"HRSBF_GPU_LOOP_{carimbo_exportacao}.zip")
with zipfile.ZipFile(zip_path,"w",compression=zipfile.ZIP_DEFLATED) as arquivo:
    for nome in ["pesquisa_fibras_loop.py","verify_fibras_loop.py","validacao_cpu_gpu.json"]:
        if Path(nome).exists(): arquivo.write(nome)
    for pasta in pastas:
        for caminho in pasta.rglob("*"):
            if caminho.is_file(): arquivo.write(caminho)
print("ZIP PRONTO:",zip_path, zip_path.stat().st_size,"bytes")
from google.colab import files
files.download(str(zip_path))


## Roteiro para o Gemini do Colab

Audite este notebook como revisor adversarial. Não altere os critérios matemáticos sem justificar. 1) Confirme a enumeração de órbitas distintas e a anulação diagonal em grau ímpar. 2) Execute os controles n=8 na CPU e na GPU; compare todas as fibras e a derivada com avaliação direta da ANF. 3) Confirme que TF32 está desativado e que as somas inteiras antes da redução módulo 2 estão no intervalo exato de float32. 4) Valide uma testemunha de exclusão por reconstrução direta, sem importar tabelas calculadas pela GPU. 5) Reteste cada cobertura aprendida usando outra semente; registre qualquer sobrevivente e procure novas fibras que o excluam. 6) Se houver candidato com todas as fibras e derivada balanceadas, calcule TODO o espectro de Walsh antes de dizer que é bent. 7) Nunca apresente amostra, cobertura finita ou opinião de IA como prova universal. Retorne comandos executados, GPU observada, contagens, arquivos e erros concretos. Não afirme que usou GPU se o backend registrar CPU.
